# 01 · Raw (messy) data profile
**Project:** Sales Variance FP&A Dashboard

This notebook only *looks at* the raw source file `data/raw/sales_raw_messy_csv.xlsx` so you can see **why cleaning is needed**. It changes nothing.
The cleaning itself is done by `scripts/clean_sales_data.py` (see notebook 02).

In [1]:
from pathlib import Path
import warnings
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 40

# Works whether Jupyter starts in notebooks/ or in the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "sales_raw_messy_csv.xlsx"
CLEAN = ROOT / "data" / "processed" / "Cleaned_Sales_Data.csv"
ENRICHED = ROOT / "data" / "processed" / "Sales_Variance_Enriched_Data.csv"
WORKBOOK = ROOT / "outputs" / "Sales_Variance_Analysis.xlsx"

## 1. Load the raw file
`keep_default_na=False` stops pandas from silently turning the text `NA` (North America) into a missing value.

In [2]:
raw = pd.read_excel(RAW, keep_default_na=False, na_values=[""])
print(f"Rows: {len(raw):,}   Columns: {raw.shape[1]}")
raw.head()

Rows: 26,956   Columns: 16


,Date,Year,Month,Month_Name,Region,Category,Product,Target_Qty,Actual_Qty,Unit_Price_USD,Unit_Cost_USD,Target_Sales_USD,Actual_Sales_USD,Gross_Margin_USD,Gross_Margin_Pct,Notes
0,2024-01-01,2024,1,January,North America,Electronics,Wireless Earbuds,142,132.00,175.50,119.54,"24,921.00","23,166.00","7,386.72",31.89,NaN
1,2024-01-01,2024,1,January,North America,Electronics,wireless earbuds,161,174.00,49.51,26.52,"7,971.11","8,614.74","4,000.26",46.44,NaN
2,2024-01-01,2024,1,January,NA,Electronics,Wireless Earbuds,127,112.00,149.48,80.01,"18,983.96","16,741.76","7,780.64",46.47,NaN
3,2024-01-01,2024,1,January,N. America,Electronics,Wireless Earbuds,127,120.00,186.15,107.47,"23,641.05","22,338.00","9,441.60",42.27,NaN
4,2024-01-01,2024,1,January,N. America,Electronics,Wireless Earbuds,231,234.00,98.77,52.60,"22,815.87","23,112.18","10,803.78",46.74,NaN


## 2. Missing values

In [3]:
missing = raw.isna().sum().rename("missing")
missing[missing > 0].to_frame().assign(pct=lambda d: d["missing"] / len(raw) * 100)

,missing,pct
Region,1163,4.31
Actual_Qty,1229,4.56
Unit_Price_USD,1167,4.33
Unit_Cost_USD,1115,4.14
Gross_Margin_USD,1224,4.54
Notes,26256,97.40


## 3. Duplicates

In [4]:
print(f"Exact duplicate rows: {raw.duplicated().sum():,}")

Exact duplicate rows: 716


## 4. Inconsistent labels

In [5]:
region_counts = raw["Region"].fillna("(blank)").value_counts()
print(f"Distinct non-blank Region labels: {raw['Region'].dropna().nunique()}  (the clean data has 5), plus {raw['Region'].isna().sum():,} blanks")
region_counts

Distinct non-blank Region labels: 22  (the clean data has 5), plus 1,163 blanks


Region
Middle East      1327
EU               1326
LATAM            1318
LatinAmerica     1304
Latin America    1302
Europe           1301
Europa           1284
ME               1281
Middle East      1276
MiddleEast       1265
europe           1258
LatAm            1254
(blank)          1163
N. America       1094
AsiaPac          1082
Northamerica     1070
North-America    1040
Asia-Pacific     1031
Asia Pacific     1024
APAC             1013
Asia Pacific     1005
North America    1002
NA                936
Name: count, dtype: int64

In [6]:
prod = raw["Product"].dropna().astype(str)
print(f"Distinct Product spellings: {prod.nunique()}")
print(f"Distinct products after trimming spaces and ignoring case: {prod.str.strip().str.lower().nunique()}")

Distinct Product spellings: 81
Distinct products after trimming spaces and ignoring case: 27


## 5. Impossible or extreme quantities

In [7]:
q = pd.to_numeric(raw["Actual_Qty"], errors="coerce")
print(f"Actual_Qty blank: {q.isna().sum():,}")
print(f"Actual_Qty negative: {(q < 0).sum():,}")
print(f"Actual_Qty above 3x the median ({q.median():,.0f}): {(q > 3 * q.median()).sum():,}")

Actual_Qty blank: 1,229
Actual_Qty negative: 212
Actual_Qty above 3x the median (150): 387


**Next:** notebook 02 shows what the cleaning script did about each of these.